# Feature Matching — Three Core Tasks

This notebook implements three tasks using **SIFT local features** and **OpenCV**:

| # | Task | Method |
|---|------|--------|
| 1 | **2D Image Registration** | SIFT + FLANN + RANSAC affine estimation → warp |
| 2 | **Affine Transformations** | Decomposed translation / rotation / scale / shear on a real image |
| 3 | **Object Recognition in a Scene** | SIFT + FLANN + RANSAC homography → bounding polygon |

All images are downloaded automatically from public URLs — no local files needed (Colab-compatible).

In [ ]:
import os
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Image download helper ────────────────────────────────────────────────────
def fetch(url: str, filename: str) -> np.ndarray:
    """Download an image from `url` if not already cached, return as BGR ndarray."""
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)
    img = cv2.imread(filename)
    if img is None:
        raise RuntimeError(f"Could not decode downloaded file: {filename}")
    return img

# ── SIFT feature helpers ─────────────────────────────────────────────────────
def detect_sift(gray: np.ndarray):
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(gray, None)
    return kp, des

def flann_match(des1: np.ndarray, des2: np.ndarray, ratio: float = 0.75):
    """FLANN kNN match + Lowe ratio test."""
    matcher = cv2.FlannBasedMatcher({"algorithm": 1, "trees": 5}, {"checks": 50})
    raw = matcher.knnMatch(des1, des2, k=2)
    return [m for m, n in raw if m.distance < ratio * n.distance]

def show(images, titles, figsize=(16, 5), cmap=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap=cmap)
        ax.set_title(title, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Libraries loaded ✓")


---
## Task 1 — 2D Image Registration using Feature Matching

**Goal:** align a *moving* image to a *fixed* reference so the same scene content overlaps pixel-for-pixel.

**Pipeline:**
1. Detect SIFT keypoints and descriptors in both images.
2. Match descriptors with FLANN + Lowe ratio test.
3. Estimate an **affine transform** with RANSAC (robust to outliers).
4. Warp the moving image into the reference coordinate frame.
5. Blend the registered result with the reference to check alignment.

In [ ]:
# ── Download Graf image pair (graffiti wall, two different viewpoints) ─────────
# graf1.png / graf3.png are a widely-used benchmark pair for planar scene
# matching: same graffiti wall photographed from different angles + zoom levels,
# making them a realistic and well-understood registration challenge.
BASE = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/"

img_fixed  = fetch(BASE + "graf1.png", "reg_fixed.png")    # reference viewpoint
img_moving = fetch(BASE + "graf3.png", "reg_moving.png")   # rotated / zoomed viewpoint

show([img_fixed, img_moving], ["Fixed — graf1 (reference)", "Moving — graf3 (different viewpoint)"])

# ── SIFT + FLANN matching ────────────────────────────────────────────────────
g_fixed  = cv2.cvtColor(img_fixed,  cv2.COLOR_BGR2GRAY)
g_moving = cv2.cvtColor(img_moving, cv2.COLOR_BGR2GRAY)

kp1, des1 = detect_sift(g_fixed)
kp2, des2 = detect_sift(g_moving)
good = flann_match(des1, des2, ratio=0.75)

print(f"SIFT keypoints — fixed: {len(kp1)},  moving: {len(kp2)}")
print(f"Good matches after ratio test: {len(good)}")

match_img = cv2.drawMatches(
    img_fixed, kp1, img_moving, kp2, good[:60], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
plt.figure(figsize=(16, 5))
plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
plt.title(f"SIFT Feature Matches — graf pair (top 60 of {len(good)})")
plt.axis("off")
plt.tight_layout()
plt.show()

# ── RANSAC full affine estimation (6-DOF) ────────────────────────────────────
# estimateAffine2D (full 6-param) is used because graf1→graf3 involves
# significant rotation, anisotropic scale and translation — partial affine
# (4-DOF: uniform scale + rotation only) fits the pair poorly.
if len(good) < 8:
    raise RuntimeError("Not enough matches for reliable registration.")

src_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)

M_est, inlier_mask = cv2.estimateAffine2D(
    src_pts, dst_pts,
    method=cv2.RANSAC, ransacReprojThreshold=3.0,
    maxIters=5000, confidence=0.99,
)
inliers = int(inlier_mask.sum()) if inlier_mask is not None else 0
print(f"\nRANSAC inliers : {inliers} / {len(good)}")
print("Estimated 2×3 affine matrix:\n", np.round(M_est, 4))

# ── Warp moving → fixed frame ─────────────────────────────────────────────────
h_f, w_f = img_fixed.shape[:2]
registered = cv2.warpAffine(img_moving, M_est, (w_f, h_f),
                             flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
overlay = cv2.addWeighted(img_fixed, 0.5, registered, 0.5, 0)

show(
    [img_fixed, img_moving, registered, overlay],
    ["Fixed (graf1)", "Moving (graf3)", "Registered", "Overlay (50/50 blend)"],
    figsize=(20, 5),
)
print("✓ Registration complete.")


---
## Task 2 — Affine Transformations on an Image

An affine transform preserves parallelism and can express **any combination** of:

| Component | Matrix effect |
|-----------|---------------|
| Translation $(t_x, t_y)$ | shift origin |
| Rotation $\theta$ | rotate around centre |
| Scale $(s_x, s_y)$ | zoom / shrink per axis |
| Shear $sh_x$ | slant horizontally |

We demonstrate each one individually, then combine them all on a real downloaded image.

In [ ]:
# ── Download sudoku.png — clean grid structure makes geometric distortions
# immediately visible: straight lines stay straight after affine warp,
# which is perfect for visually verifying each transformation.
BASE = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/"
aff_src = fetch(BASE + "sudoku.png", "affine_src.png")
aff_src = cv2.resize(aff_src, (320, 320), interpolation=cv2.INTER_AREA)
h, w  = aff_src.shape[:2]
cx, cy = w / 2.0, h / 2.0


def warp_affine(img, M2x3, border=cv2.BORDER_REFLECT):
    return cv2.warpAffine(img, M2x3, (img.shape[1], img.shape[0]),
                          flags=cv2.INTER_LINEAR, borderMode=border)


# 1) Translation
M_translate = np.float32([[1, 0, 40],
                           [0, 1, 30]])

# 2) Rotation — 25° CCW around image centre
M_rotate = cv2.getRotationMatrix2D((cx, cy), angle=25, scale=1.0)

# 3) Anisotropic scale — sx=1.3, sy=0.75, centre-anchored
sx, sy = 1.3, 0.75
M_scale = np.float32([[sx, 0,  cx * (1 - sx)],
                       [0,  sy, cy * (1 - sy)]])

# 4) Horizontal shear — shx=0.3
shx = 0.3
M_shear = np.float32([[1,   shx, -cy * shx],
                       [0,   1,    0        ]])

# 5) Combined — rotation 15° + scale 0.9 + shear + translation
M_r  = cv2.getRotationMatrix2D((cx, cy), angle=15, scale=0.9)
M_r3 = np.vstack([M_r, [0, 0, 1]])
M_tx3 = np.array([[1, 0.12, 20],
                   [0, 1,    15],
                   [0, 0,     1]], dtype=np.float64)
M_combined = (M_tx3 @ M_r3)[:2]

transforms = [
    (M_translate, f"Translation\ntx=40, ty=30 px"),
    (M_rotate,    "Rotation\n25° CCW (centred)"),
    (M_scale,     f"Scale\nsx={sx}, sy={sy} (centred)"),
    (M_shear,     f"Shear\nshx={shx}"),
    (M_combined,  "Combined\n(rot+scale+shear+trans)"),
]

results = [aff_src] + [warp_affine(aff_src, M) for M, _ in transforms]
titles  = ["Original"] + [t for _, t in transforms]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img, title in zip(axes.ravel(), results, titles):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=11)
    ax.axis("off")
axes.ravel()[-1].set_visible(False)
plt.suptitle("Affine Transformations on a Real Image (sudoku.png)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Combined affine matrix (2×3):")
print(np.round(M_combined, 4))
print("\n✓ All five affine transforms applied.")


---
## Task 3 — Object Recognition in a Scene using Local Feature Matching

**Goal:** locate a known *object* (template) inside a larger *scene* image.

**Pipeline:**
1. Detect SIFT features on both object and scene.
2. Match with FLANN + Lowe ratio test.
3. Estimate a **homography** with RANSAC (handles perspective, unlike affine).
4. Project the object's four corners into the scene — draws the detected bounding polygon.

In [ ]:
# ── Download the classic OpenCV box + scene pair ──────────────────────────────
# box.png          — clean studio photograph of a box (the object template).
# box_in_scene.png — same box placed in a cluttered, partially occluded scene.
# This is the canonical OpenCV feature-matching demo: real object, real clutter,
# significant viewpoint change — a proper test for homography estimation.
BASE = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/"

img_obj   = fetch(BASE + "box.png",          "obj_box.png")
img_scene = fetch(BASE + "box_in_scene.png", "scene_box.png")

show([img_obj, img_scene],
     ["Object — box.png (template)", "Scene — box_in_scene.png"],
     figsize=(12, 5))

# ── SIFT detection ────────────────────────────────────────────────────────────
g_obj   = cv2.cvtColor(img_obj,   cv2.COLOR_BGR2GRAY)
g_scene = cv2.cvtColor(img_scene, cv2.COLOR_BGR2GRAY)

kp_o, des_o = detect_sift(g_obj)
kp_s, des_s = detect_sift(g_scene)

# Slightly tighter ratio (0.7) to reduce false positives in the cluttered scene
good_or = flann_match(des_o, des_s, ratio=0.7)

print(f"Object keypoints : {len(kp_o)}")
print(f"Scene  keypoints : {len(kp_s)}")
print(f"Good matches     : {len(good_or)}")

if len(good_or) < 10:
    raise RuntimeError("Too few matches — check ratio threshold.")

# ── Homography with RANSAC ────────────────────────────────────────────────────
obj_pts   = np.float32([kp_o[m.queryIdx].pt for m in good_or]).reshape(-1, 1, 2)
scene_pts = np.float32([kp_s[m.trainIdx].pt for m in good_or]).reshape(-1, 1, 2)

H_mat, h_mask = cv2.findHomography(obj_pts, scene_pts,
                                    cv2.RANSAC, ransacReprojThreshold=5.0)
h_inliers = int(h_mask.sum()) if h_mask is not None else 0
print(f"RANSAC inliers   : {h_inliers} / {len(good_or)}")

if H_mat is None:
    raise RuntimeError("Homography estimation failed.")

# ── Project object corners into scene ─────────────────────────────────────────
ho, wo = img_obj.shape[:2]
corners_obj = np.float32([[0, 0], [wo, 0], [wo, ho], [0, ho]]).reshape(-1, 1, 2)
corners_scene = cv2.perspectiveTransform(corners_obj, H_mat)

# ── Match visualisation with bounding polygon ─────────────────────────────────
inlier_list = h_mask.ravel().tolist() if h_mask is not None else None
match_vis = cv2.drawMatches(
    img_obj, kp_o, img_scene, kp_s, good_or, None,
    matchesMask=inlier_list,
    matchColor=(0, 230, 0),
    singlePointColor=(200, 0, 0),
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
offset = np.float32([[[img_obj.shape[1], 0]]])
cv2.polylines(match_vis, [np.int32(corners_scene + offset)],
              isClosed=True, color=(0, 0, 255), thickness=3, lineType=cv2.LINE_AA)

plt.figure(figsize=(18, 7))
plt.imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))
plt.title(f"Object Recognition — {h_inliers} inlier matches  |  blue polygon = detected object",
          fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

# ── Detected polygon drawn on the scene ───────────────────────────────────────
scene_marked = img_scene.copy()
cv2.polylines(scene_marked, [np.int32(corners_scene)],
              isClosed=True, color=(0, 255, 0), thickness=3, lineType=cv2.LINE_AA)
for pt, lbl in zip(corners_scene.reshape(-1, 2), ["TL", "TR", "BR", "BL"]):
    cv2.putText(scene_marked, lbl, (int(pt[0]) + 6, int(pt[1]) - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

show([img_obj, scene_marked],
     ["Object template (box.png)", "Detected location in scene (green polygon)"],
     figsize=(14, 6))

print("\n✓ Object successfully located in scene.")
print(f"  Corner coordinates (px): {np.int32(corners_scene).reshape(-1, 2).tolist()}")
